# Data preprocessing

## Objective

Prepare the cleaned Telco Customer Churn dataset for machine learning using a reproducible and leakage-safe preprocessing workflow.

This notebook will:

- separate the target variable from the predictive features;
- create stratified training and test sets;
- identify numerical, binary, and categorical features;
- handle missing numerical values;
- encode categorical variables;
- preserve preprocessing consistency between training and test data;
- build a reusable `ColumnTransformer` for later modeling.

All preprocessing operations that learn information from the data will be fitted only on the training set to prevent data leakage.

In [43]:
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
import numpy as np
from IPython.display import display
import joblib

In [44]:
PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_PATH = DATA_DIR / "raw" / "WA_Fn-UseC_-Telco-Customer-Churn.csv"
if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found at {RAW_DATA_PATH.resolve()}")
dataset = pd.read_csv(RAW_DATA_PATH)

In [45]:
dataset

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,6840-RESVB,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,...,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.5,No
7039,2234-XADUH,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,...,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.9,No
7040,4801-JZAZL,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,No
7041,8361-LTMKD,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.6,Yes


In [46]:
dataset["TotalCharges"] = pd.to_numeric(dataset["TotalCharges"],errors="coerce")

In [47]:
print(dataset["TotalCharges"].dtype)
print("Missing TotalCharges:", dataset["TotalCharges"].isna().sum())

float64
Missing TotalCharges: 11


## Define features and target

The dataset is separated into predictive features (`X`) and the target variable (`y`).

- `X` contains the customer attributes that will be used by the machine learning models to make predictions.
- `y` contains the `Churn` variable, which represents the outcome the models must predict.

The target is encoded as:

- `0`: the customer did not churn;
- `1`: the customer churned.

The `customerID` column is excluded from the predictive features because it is only an identifier and does not represent customer behavior or service characteristics.

In [48]:
X = dataset.drop(columns=["Churn","customerID"])
X

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.50
7039,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.90
7040,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,No,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45
7041,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.60


In [49]:
print("Validation...")
print(f"Columns into list: {X.columns.to_list()}")
print(f"Columns len: {len(X.columns.to_list())}")

Validation...
Columns into list: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges']
Columns len: 19


In [50]:
y = dataset["Churn"].map({'Yes':1,'No':0})
print("Validation...")
print(f"Value counts:\n{y.value_counts()}")
print("-"*50)
print(f"Value counts from dataset:\n{dataset.value_counts("Churn")}")

Validation...
Value counts:
Churn
0    5174
1    1869
Name: count, dtype: int64
--------------------------------------------------
Value counts from dataset:
Churn
No     5174
Yes    1869
Name: count, dtype: int64


In [51]:
assert "customerID" not in X.columns
assert "Churn" not in X.columns
assert X.shape == (7043, 19)

assert y.isna().sum() == 0
assert set(y.unique()) == {0, 1}
assert len(X) == len(y)

print("X and y were created successfully.")

X and y were created successfully.


### Validation results

The feature matrix `X` contains 7,043 customer records and 19 predictive features.

The `customerID` identifier and the `Churn` target were correctly excluded from `X`. The target vector `y` was successfully encoded as:

- `0`: No Churn
- `1`: Churn

The target contains 5,174 customers in class `0` and 1,869 customers in class `1`. No missing values were introduced during the encoding process, and the number of rows in `X` matches the number of observations in `y`.

These validation checks confirm that the features and target were created correctly and are ready for the train-test split.

## Train/test split

The dataset is divided into separate training and test sets before fitting any preprocessing transformations.

- The **training set** contains 80% of the observations and will be used to fit preprocessing steps, train models, perform cross-validation, and select hyperparameters.
- The **test set** contains the remaining 20% and will remain untouched until the final evaluation of the selected model.

A stratified split is used to preserve approximately the same proportion of churned and non-churned customers in both sets. A fixed `random_state` makes the split reproducible.

Creating the split before fitting the preprocessing pipeline prevents data leakage and provides a more reliable estimate of model performance on unseen customers.

In [52]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=.2,random_state=42,stratify=y)

In [53]:
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print("-"*50)

print("\nTraining target proportions:")
display(y_train.value_counts(normalize=True).round(4)*100)
print("Test target proportions:")
display(y_test.value_counts(normalize=True).round(4)*100)

X_train shape: (5634, 19)
X_test shape: (1409, 19)
y_train shape: (5634,)
y_test shape: (1409,)
--------------------------------------------------

Training target proportions:


Churn
0    73.46
1    26.54
Name: proportion, dtype: float64

Test target proportions:


Churn
0    73.46
1    26.54
Name: proportion, dtype: float64

### Split validation

The dataset was successfully divided into:

- **Training set:** 5,634 customers and 19 predictor features.
- **Test set:** 1,409 customers and 19 predictor features.

The target arrays contain the corresponding 5,634 training labels and 1,409 test labels, confirming that each observation has a target value.

Both sets preserve the original target distribution:

- approximately **73.5%** of customers did not churn (`0`);
- approximately **26.5%** of customers churned (`1`).

Therefore, `stratify=y` worked as intended. The training and test sets have comparable class distributions, reducing the possibility that model evaluation will be distorted by an unrepresentative split.

The test set will now remain untouched until the final evaluation of the selected model. All preprocessing transformations will be fitted using only the training data.

## Feature type identification

The predictor features are separated into numerical and categorical groups because they require different preprocessing transformations.

- **Numerical features** contain quantitative values and will be imputed and standardized when required.
- **Categorical features** contain nominal categories and will be imputed and converted into numerical columns using one-hot encoding.

Feature types are identified using only `X_train`. This keeps the preprocessing workflow based exclusively on the training data and helps prevent accidental use of the test set.

In [54]:
PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DATA_PATH = DATA_DIR / "processed" / "column_audit.csv"
if not PROCESSED_DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found at {PROCESSED_DATA_PATH.resolve()}")
column_audit = pd.read_csv(PROCESSED_DATA_PATH, index_col=0)
column_audit

,column,dtype,unique_count,possible_role
0,customerID,str,7043,identifier
1,gender,str,2,binary categorical
2,SeniorCitizen,int64,2,binary categorical stored as integer
3,Partner,str,2,binary categorical
4,Dependents,str,2,binary categorical
5,tenure,int64,73,numeric
6,PhoneService,str,2,binary categorical
7,MultipleLines,str,3,multiclass categorical
8,InternetService,str,3,multiclass categorical
9,OnlineSecurity,str,3,multiclass categorical


In [55]:
numerical_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
binary_features = ["SeniorCitizen"]
categorical_features = X_train.select_dtypes(include="str").columns.to_list()

In [56]:
all_classified_features = (numerical_features + binary_features + categorical_features)

assert len(all_classified_features) == X_train.shape[1]
assert set(all_classified_features) == set(X_train.columns)

print("Numerical:", numerical_features)
print("Binary:", binary_features)
print("Categorical:", categorical_features)
print("\nAll features were classified successfully.")

Numerical: ['tenure', 'MonthlyCharges', 'TotalCharges']
Binary: ['SeniorCitizen']
Categorical: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']

All features were classified successfully.


In [57]:
column_audit = column_audit[(column_audit["column"] != "customerID") & (column_audit["column"] != "Churn")]
column_audit.loc[column_audit["column"] == "TotalCharges", "dtype"] = "float64"

In [58]:
binary_feature_preprocess = column_audit[(column_audit["unique_count"] == 2) & (column_audit["dtype"] != "int64")]
binary_feature_preprocess = binary_feature_preprocess["column"].to_list()

multiclass_feature_preprocess = column_audit[
    (column_audit["unique_count"] > 2) &
    (column_audit["dtype"] != "int64") &
    (column_audit["dtype"] != "float64")
]
multiclass_feature_preprocess = multiclass_feature_preprocess["column"].to_list()

print(f'Binary categorizal:\n{binary_feature_preprocess}')
print(f'Multicass categorical:\n{multiclass_feature_preprocess}')

Binary categorizal:
['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
Multicass categorical:
['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaymentMethod']


In [59]:
assert set(categorical_features) == set(
    binary_feature_preprocess
    + multiclass_feature_preprocess
), "Some categorical features were not classified correctly."

print("All categorical features were classified successfully.")

All categorical features were classified successfully.


## Preprocessing pipeline construction

A reusable preprocessing workflow is created to transform the training and test data consistently and without data leakage.

The workflow includes the following operations:

- **Missing-value imputation:** missing values in `TotalCharges` are replaced with `0`, representing new customers with no accumulated billing history.
- **Numerical scaling:** `tenure`, `MonthlyCharges`, and `TotalCharges` are standardized using `StandardScaler`.
- **Categorical encoding:** categorical features are converted into numerical indicator columns using one-hot encoding.
- **Column-specific transformation:** a `ColumnTransformer` applies the numerical and categorical pipelines only to their corresponding feature groups.

The preprocessor will be fitted exclusively on `X_train`. The fitted transformations will then be applied to `X_test` without refitting, preventing information from the test set from influencing the preprocessing process.

In [60]:
numerical_pipeline = Pipeline(
    steps=[
        ("imputer",SimpleImputer(strategy="constant",fill_value=0)),
        ("scaler",StandardScaler())
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

In [61]:
preprocessor = ColumnTransformer(
    transformers=[
        ("numerical_pipeline", numerical_pipeline, numerical_features),
        ("categorical_pipeline", categorical_pipeline, (binary_feature_preprocess + multiclass_feature_preprocess)),
        ("binary_passthrough","passthrough",binary_features)
    ]
)

### Fit and transform the data

The preprocessor is fitted using only the training data.

`fit_transform()` learns the numerical scaling parameters, categorical structure, and missing-value transformation from `X_train`. The fitted preprocessor is subsequently applied to `X_test` using `transform()` only.

This guarantees that both datasets receive the same transformations while keeping the test set isolated from the training process.

In [62]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

### Preprocessing validation

The preprocessing workflow was completed successfully using a `ColumnTransformer`.

The numerical pipeline replaced missing values in `TotalCharges` with `0` and standardized the numerical variables. The categorical pipeline converted the selected categorical features into binary indicator columns using one-hot encoding. The `SeniorCitizen` feature was preserved without modification because it was already encoded as a binary numerical variable.

The preprocessor was fitted exclusively on `X_train` and then applied to `X_test` without refitting. This prevents information from the test set from influencing the preprocessing process.

The processed training set contains 5,634 observations and 44 model-ready features. The processed test set contains 1,409 observations with the same 44-feature structure.

No missing values remain in either transformed dataset, confirming that the imputation strategy was applied correctly. The matching number of columns also confirms that the same fitted preprocessing structure was used consistently for both training and test data.

The increase from 19 original predictor variables to 44 transformed features is mainly caused by one-hot encoding, which creates separate binary columns for categorical values.

In [63]:
assert X_train.shape[0] == X_train_processed.shape[0], "Lost row in train"
assert X_test.shape[0] == X_test_processed.shape[0], "Lost row in test"
assert X_train_processed.shape[1] == X_test_processed.shape[1], "Train and test have different columns"

In [64]:
assert not np.isnan(X_train_processed).any(), "There are NaN in train"
assert not np.isnan(X_test_processed).any(), "There are NaN in test"

In [65]:
print("Train processed shape:", X_train_processed.shape)
print("Test processed shape:", X_test_processed.shape)

print("Missing values in train:", np.isnan(X_train_processed).sum())
print("Missing values in test:", np.isnan(X_test_processed).sum())

Train processed shape: (5634, 45)
Test processed shape: (1409, 45)
Missing values in train: 0
Missing values in test: 0


In [66]:
feature_names = preprocessor.get_feature_names_out()

X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_test_processed_df = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)
print("X_train processed:")
display(X_train_processed_df)
print("X_test processed:")
display(X_test_processed_df)

X_train processed:


,numerical_pipeline__tenure,numerical_pipeline__MonthlyCharges,numerical_pipeline__TotalCharges,categorical_pipeline__gender_Female,categorical_pipeline__gender_Male,categorical_pipeline__Partner_No,categorical_pipeline__Partner_Yes,categorical_pipeline__Dependents_No,categorical_pipeline__Dependents_Yes,categorical_pipeline__PhoneService_No,...,categorical_pipeline__StreamingMovies_No internet service,categorical_pipeline__StreamingMovies_Yes,categorical_pipeline__Contract_Month-to-month,categorical_pipeline__Contract_One year,categorical_pipeline__Contract_Two year,categorical_pipeline__PaymentMethod_Bank transfer (automatic),categorical_pipeline__PaymentMethod_Credit card (automatic),categorical_pipeline__PaymentMethod_Electronic check,categorical_pipeline__PaymentMethod_Mailed check,binary_passthrough__SeniorCitizen
3738,0.102371,-0.521976,-0.262257,0.0,1.0,1.0,0.0,1.0,0.0,1.0,...,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3151,-0.711743,0.337478,-0.503635,0.0,1.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4860,-0.793155,-0.809013,-0.749883,0.0,1.0,0.0,1.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
3867,-0.263980,0.284384,-0.172722,1.0,0.0,0.0,1.0,1.0,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
3810,-1.281624,-0.676279,-0.989374,0.0,1.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6303,1.567778,1.470695,2.373129,1.0,0.0,0.0,1.0,1.0,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
6227,-1.240918,-0.626504,-0.973665,0.0,1.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
4673,-0.304686,1.256662,0.158344,1.0,0.0,1.0,0.0,1.0,0.0,0.0,...,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
2710,-0.345392,-1.477661,-0.797075,1.0,0.0,0.0,1.0,1.0,0.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0


X_test processed:


,numerical_pipeline__tenure,numerical_pipeline__MonthlyCharges,numerical_pipeline__TotalCharges,categorical_pipeline__gender_Female,categorical_pipeline__gender_Male,categorical_pipeline__Partner_No,categorical_pipeline__Partner_Yes,categorical_pipeline__Dependents_No,categorical_pipeline__Dependents_Yes,categorical_pipeline__PhoneService_No,...,categorical_pipeline__StreamingMovies_No internet service,categorical_pipeline__StreamingMovies_Yes,categorical_pipeline__Contract_Month-to-month,categorical_pipeline__Contract_One year,categorical_pipeline__Contract_Two year,categorical_pipeline__PaymentMethod_Bank transfer (automatic),categorical_pipeline__PaymentMethod_Credit card (automatic),categorical_pipeline__PaymentMethod_Electronic check,categorical_pipeline__PaymentMethod_Mailed check,binary_passthrough__SeniorCitizen
437,1.608483,1.629976,2.706828,0.0,1.0,0.0,1.0,0.0,1.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
2280,-0.996684,1.168725,-0.610260,1.0,0.0,1.0,0.0,1.0,0.0,0.0,...,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
2235,0.346606,0.445324,0.400116,1.0,0.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
4460,-0.589626,0.440347,-0.364451,0.0,1.0,0.0,1.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3761,1.608483,0.588013,1.588421,1.0,0.0,0.0,1.0,1.0,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5143,0.672252,0.738999,0.897615,1.0,0.0,0.0,1.0,0.0,1.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
4439,-0.182569,-1.480980,-0.794815,0.0,1.0,0.0,1.0,0.0,1.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
3857,-1.118801,-1.469365,-0.967873,0.0,1.0,1.0,0.0,1.0,0.0,0.0,...,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
4758,0.957192,-1.500890,-0.547360,1.0,0.0,1.0,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0


In [67]:
display(X_train_processed_df.describe().round(5))
display(X_test_processed_df.describe().round(5))

,numerical_pipeline__tenure,numerical_pipeline__MonthlyCharges,numerical_pipeline__TotalCharges,categorical_pipeline__gender_Female,categorical_pipeline__gender_Male,categorical_pipeline__Partner_No,categorical_pipeline__Partner_Yes,categorical_pipeline__Dependents_No,categorical_pipeline__Dependents_Yes,categorical_pipeline__PhoneService_No,...,categorical_pipeline__StreamingMovies_No internet service,categorical_pipeline__StreamingMovies_Yes,categorical_pipeline__Contract_Month-to-month,categorical_pipeline__Contract_One year,categorical_pipeline__Contract_Two year,categorical_pipeline__PaymentMethod_Bank transfer (automatic),categorical_pipeline__PaymentMethod_Credit card (automatic),categorical_pipeline__PaymentMethod_Electronic check,categorical_pipeline__PaymentMethod_Mailed check,binary_passthrough__SeniorCitizen
count,5634.00000,5634.00000,5634.00000,5634.00000,5634.00000,5634.00000,5634.00000,5634.00000,5634.00000,5634.00000,...,5634.00000,5634.00000,5634.00000,5634.00000,5634.00000,5634.00000,5634.00000,5634.00000,5634.00000,5634.00000
mean,-0.00000,-0.00000,0.00000,0.49716,0.50284,0.51562,0.48438,0.70199,0.29801,0.09922,...,0.21548,0.39102,0.55059,0.20820,0.24121,0.22080,0.21530,0.33564,0.22826,0.16329
std,1.00009,1.00009,1.00009,0.50004,0.50004,0.49980,0.49980,0.45743,0.45743,0.29898,...,0.41119,0.48802,0.49748,0.40606,0.42786,0.41482,0.41107,0.47226,0.41975,0.36967
min,-1.32233,-1.54403,-1.00892,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,...,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000
25%,-0.95598,-0.97120,-0.83210,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,...,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000
50%,-0.14186,0.18483,-0.39684,0.00000,1.00000,1.00000,0.00000,1.00000,0.00000,0.00000,...,0.00000,0.00000,1.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000
75%,0.91649,0.83191,0.67419,1.00000,1.00000,1.00000,1.00000,1.00000,1.00000,0.00000,...,0.00000,1.00000,1.00000,0.00000,0.00000,0.00000,0.00000,1.00000,0.00000,0.00000
max,1.60848,1.78594,2.80187,1.00000,1.00000,1.00000,1.00000,1.00000,1.00000,1.00000,...,1.00000,1.00000,1.00000,1.00000,1.00000,1.00000,1.00000,1.00000,1.00000,1.00000


,numerical_pipeline__tenure,numerical_pipeline__MonthlyCharges,numerical_pipeline__TotalCharges,categorical_pipeline__gender_Female,categorical_pipeline__gender_Male,categorical_pipeline__Partner_No,categorical_pipeline__Partner_Yes,categorical_pipeline__Dependents_No,categorical_pipeline__Dependents_Yes,categorical_pipeline__PhoneService_No,...,categorical_pipeline__StreamingMovies_No internet service,categorical_pipeline__StreamingMovies_Yes,categorical_pipeline__Contract_Month-to-month,categorical_pipeline__Contract_One year,categorical_pipeline__Contract_Two year,categorical_pipeline__PaymentMethod_Bank transfer (automatic),categorical_pipeline__PaymentMethod_Credit card (automatic),categorical_pipeline__PaymentMethod_Electronic check,categorical_pipeline__PaymentMethod_Mailed check,binary_passthrough__SeniorCitizen
count,1409.00000,1409.00000,1409.00000,1409.00000,1409.00000,1409.00000,1409.00000,1409.00000,1409.00000,1409.00000,...,1409.00000,1409.00000,1409.00000,1409.00000,1409.00000,1409.00000,1409.00000,1409.00000,1409.00000,1409.00000
mean,-0.02318,-0.02791,-0.04299,0.48758,0.51242,0.52236,0.47764,0.69411,0.30589,0.08730,...,0.22143,0.37544,0.54862,0.21292,0.23847,0.21292,0.21930,0.33641,0.23137,0.15756
std,0.99834,0.99213,0.97215,0.50002,0.50002,0.49968,0.49968,0.46095,0.46095,0.28237,...,0.41536,0.48441,0.49781,0.40951,0.42630,0.40951,0.41392,0.47265,0.42186,0.36446
min,-1.32233,-1.54901,-1.00892,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,...,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000
25%,-0.99668,-0.98655,-0.85091,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,...,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000
50%,-0.18257,0.16658,-0.39701,0.00000,1.00000,1.00000,0.00000,1.00000,0.00000,0.00000,...,0.00000,0.00000,1.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000
75%,0.91649,0.81366,0.53469,1.00000,1.00000,1.00000,1.00000,1.00000,1.00000,0.00000,...,0.00000,1.00000,1.00000,0.00000,0.00000,0.00000,0.00000,1.00000,0.00000,0.00000
max,1.60848,1.72123,2.71933,1.00000,1.00000,1.00000,1.00000,1.00000,1.00000,1.00000,...,1.00000,1.00000,1.00000,1.00000,1.00000,1.00000,1.00000,1.00000,1.00000,1.00000


In [68]:
X_train_processed_df.columns.to_list()

['numerical_pipeline__tenure',
 'numerical_pipeline__MonthlyCharges',
 'numerical_pipeline__TotalCharges',
 'categorical_pipeline__gender_Female',
 'categorical_pipeline__gender_Male',
 'categorical_pipeline__Partner_No',
 'categorical_pipeline__Partner_Yes',
 'categorical_pipeline__Dependents_No',
 'categorical_pipeline__Dependents_Yes',
 'categorical_pipeline__PhoneService_No',
 'categorical_pipeline__PhoneService_Yes',
 'categorical_pipeline__PaperlessBilling_No',
 'categorical_pipeline__PaperlessBilling_Yes',
 'categorical_pipeline__MultipleLines_No',
 'categorical_pipeline__MultipleLines_No phone service',
 'categorical_pipeline__MultipleLines_Yes',
 'categorical_pipeline__InternetService_DSL',
 'categorical_pipeline__InternetService_Fiber optic',
 'categorical_pipeline__InternetService_No',
 'categorical_pipeline__OnlineSecurity_No',
 'categorical_pipeline__OnlineSecurity_No internet service',
 'categorical_pipeline__OnlineSecurity_Yes',
 'categorical_pipeline__OnlineBackup_No',


In [69]:
X_test_processed_df.columns.to_list()

['numerical_pipeline__tenure',
 'numerical_pipeline__MonthlyCharges',
 'numerical_pipeline__TotalCharges',
 'categorical_pipeline__gender_Female',
 'categorical_pipeline__gender_Male',
 'categorical_pipeline__Partner_No',
 'categorical_pipeline__Partner_Yes',
 'categorical_pipeline__Dependents_No',
 'categorical_pipeline__Dependents_Yes',
 'categorical_pipeline__PhoneService_No',
 'categorical_pipeline__PhoneService_Yes',
 'categorical_pipeline__PaperlessBilling_No',
 'categorical_pipeline__PaperlessBilling_Yes',
 'categorical_pipeline__MultipleLines_No',
 'categorical_pipeline__MultipleLines_No phone service',
 'categorical_pipeline__MultipleLines_Yes',
 'categorical_pipeline__InternetService_DSL',
 'categorical_pipeline__InternetService_Fiber optic',
 'categorical_pipeline__InternetService_No',
 'categorical_pipeline__OnlineSecurity_No',
 'categorical_pipeline__OnlineSecurity_No internet service',
 'categorical_pipeline__OnlineSecurity_Yes',
 'categorical_pipeline__OnlineBackup_No',


In [70]:
encoded_columns = [
    column
    for column in X_train_processed_df.columns
    if column.startswith("categorical_pipeline__")
]

print("Number of encoded columns:", len(encoded_columns))
print("\nEncoded columns:")
print(encoded_columns)

Number of encoded columns: 41

Encoded columns:
['categorical_pipeline__gender_Female', 'categorical_pipeline__gender_Male', 'categorical_pipeline__Partner_No', 'categorical_pipeline__Partner_Yes', 'categorical_pipeline__Dependents_No', 'categorical_pipeline__Dependents_Yes', 'categorical_pipeline__PhoneService_No', 'categorical_pipeline__PhoneService_Yes', 'categorical_pipeline__PaperlessBilling_No', 'categorical_pipeline__PaperlessBilling_Yes', 'categorical_pipeline__MultipleLines_No', 'categorical_pipeline__MultipleLines_No phone service', 'categorical_pipeline__MultipleLines_Yes', 'categorical_pipeline__InternetService_DSL', 'categorical_pipeline__InternetService_Fiber optic', 'categorical_pipeline__InternetService_No', 'categorical_pipeline__OnlineSecurity_No', 'categorical_pipeline__OnlineSecurity_No internet service', 'categorical_pipeline__OnlineSecurity_Yes', 'categorical_pipeline__OnlineBackup_No', 'categorical_pipeline__OnlineBackup_No internet service', 'categorical_pipelin

In [71]:
assert X_train_processed_df.columns.equals(X_test_processed_df.columns)
assert "binary_passthrough__SeniorCitizen" in feature_names
assert len(feature_names) == 45
print("Train and test columns match.")

Train and test columns match.


In [72]:
encoded_train_values = X_train_processed_df[encoded_columns]
encoded_test_values = X_test_processed_df[encoded_columns]

assert encoded_train_values.isin([0, 1]).all().all()
assert encoded_test_values.isin([0, 1]).all().all()

print("All one-hot encoded values are binary.")

All one-hot encoded values are binary.


In [73]:
fitted_encoder = (
    preprocessor
    .named_transformers_["categorical_pipeline"]
    .named_steps["onehot"]
)

for feature, categories in zip(
    (binary_feature_preprocess + multiclass_feature_preprocess),
    fitted_encoder.categories_
):
    print(f"{feature}: {len(categories)} categories -> {list(categories)}")

gender: 2 categories -> ['Female', 'Male']
Partner: 2 categories -> ['No', 'Yes']
Dependents: 2 categories -> ['No', 'Yes']
PhoneService: 2 categories -> ['No', 'Yes']
PaperlessBilling: 2 categories -> ['No', 'Yes']
MultipleLines: 3 categories -> ['No', 'No phone service', 'Yes']
InternetService: 3 categories -> ['DSL', 'Fiber optic', 'No']
OnlineSecurity: 3 categories -> ['No', 'No internet service', 'Yes']
OnlineBackup: 3 categories -> ['No', 'No internet service', 'Yes']
DeviceProtection: 3 categories -> ['No', 'No internet service', 'Yes']
TechSupport: 3 categories -> ['No', 'No internet service', 'Yes']
StreamingTV: 3 categories -> ['No', 'No internet service', 'Yes']
StreamingMovies: 3 categories -> ['No', 'No internet service', 'Yes']
Contract: 3 categories -> ['Month-to-month', 'One year', 'Two year']
PaymentMethod: 4 categories -> ['Bank transfer (automatic)', 'Credit card (automatic)', 'Electronic check', 'Mailed check']


In [74]:
expected_encoded_columns = sum(
    len(categories)
    for categories in fitted_encoder.categories_
)

assert len(encoded_columns) == expected_encoded_columns

print(
    "The number of generated columns matches "
    "the categories learned by OneHotEncoder."
)

The number of generated columns matches the categories learned by OneHotEncoder.


### One-hot encoding validation results

The one-hot encoding validation was completed successfully.

The encoded training and test datasets contain the same categorical columns in the same order. All generated indicator values are binary, and the total number of encoded columns matches the number of categories learned from the training data.

These results confirm that the categorical transformations were applied consistently and that the processed categorical features are ready for modeling.

## Leakage validation

This section verifies that the preprocessing workflow did not allow information from the test set or target variable to influence the training process.

The validation checks confirm that:

- the target variable is excluded from the predictor matrices;
- the training and test sets contain different observations;
- the preprocessor was fitted only on `X_train`;
- `X_test` was transformed using the already-fitted preprocessor;
- preprocessing preserved the number of observations in both datasets.

In [75]:
# 1. The target must not be included in the predictors
assert "Churn" not in X_train.columns
assert "Churn" not in X_test.columns

# 2. Train and test observations must not overlap
assert set(X_train.index).isdisjoint(set(X_test.index))

# 3. The number of observations must be preserved
assert X_train_processed_df.shape[0] == X_train.shape[0]
assert X_test_processed_df.shape[0] == X_test.shape[0]

# 4. The preprocessor must already be fitted
assert hasattr(preprocessor, "transformers_")

print("Leakage validation checks passed successfully.")

Leakage validation checks passed successfully.


### Leakage validation results

All leakage validation checks passed successfully.

The target variable was excluded from the predictor matrices, and no observations were shared between the training and test sets. The preprocessing transformations also preserved the original number of observations.

The preprocessor was fitted on `X_train`, while `X_test` was transformed using the already-fitted preprocessing structure. This maintains the required separation between training and test data and reduces the risk of data leakage.

## Transformation validation

This section verifies that the preprocessing pipeline produced consistent and model-ready datasets.

The validation checks confirm that:

- the number of observations was preserved after transformation;
- the training and test sets contain the same transformed features;
- the feature columns appear in the same order;
- all transformed values are numerical;
- no missing or infinite values remain;
- one-hot encoded columns contain only binary values.

In [76]:
# Row counts must be preserved
assert X_train_processed_df.shape[0] == X_train.shape[0]
assert X_test_processed_df.shape[0] == X_test.shape[0]

# Train and test must have the same transformed structure
assert X_train_processed_df.shape[1] == X_test_processed_df.shape[1]
assert X_train_processed_df.columns.equals(
    X_test_processed_df.columns
)

# All transformed columns must be numerical
assert all(
    np.issubdtype(dtype, np.number)
    for dtype in X_train_processed_df.dtypes
)

assert all(
    np.issubdtype(dtype, np.number)
    for dtype in X_test_processed_df.dtypes
)

# No missing values
assert X_train_processed_df.isna().sum().sum() == 0
assert X_test_processed_df.isna().sum().sum() == 0

# No infinite values
assert np.isfinite(X_train_processed_df.to_numpy()).all()
assert np.isfinite(X_test_processed_df.to_numpy()).all()

print("Transformation validation checks passed successfully.")

Transformation validation checks passed successfully.


In [77]:
encoded_columns = [
    column
    for column in X_train_processed_df.columns
    if column.startswith("categorical_pipeline__")
]

assert len(encoded_columns) > 0

assert (
    X_train_processed_df[encoded_columns]
    .isin([0, 1])
    .all()
    .all()
)

assert (
    X_test_processed_df[encoded_columns]
    .isin([0, 1])
    .all()
    .all()
)

print("One-hot encoded columns contain only 0 and 1.")

One-hot encoded columns contain only 0 and 1.


### Transformation validation results

All transformation checks passed successfully.

The preprocessing workflow preserved the number of observations in both datasets. The transformed training and test sets contain the same 44 numerical features in the same order.

No missing or infinite values remain, and the one-hot encoded columns contain only binary values. These results confirm that the transformed datasets are consistent and ready for machine learning.

## Save feature metadata

The transformed feature names and their positions are saved as metadata.

Preserving this information ensures that later modeling, interpretation, and deployment stages can identify the exact columns generated by the fitted preprocessing pipeline and maintain their expected order.

In [78]:
feature_metadata = pd.DataFrame({
    "feature_position": range(len(feature_names)),
    "feature_name": feature_names
})

feature_metadata.head(10)

,feature_position,feature_name
0,0,numerical_pipeline__tenure
1,1,numerical_pipeline__MonthlyCharges
2,2,numerical_pipeline__TotalCharges
3,3,categorical_pipeline__gender_Female
4,4,categorical_pipeline__gender_Male
5,5,categorical_pipeline__Partner_No
6,6,categorical_pipeline__Partner_Yes
7,7,categorical_pipeline__Dependents_No
8,8,categorical_pipeline__Dependents_Yes
9,9,categorical_pipeline__PhoneService_No


In [79]:
FEATURE_METADATA_PATH = (
    DATA_DIR
    / "processed"
    / "feature_metadata.csv"
)

FEATURE_METADATA_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

feature_metadata.to_csv(
    FEATURE_METADATA_PATH,
    index=False
)

print(
    "Feature metadata saved to:",
    FEATURE_METADATA_PATH.resolve()
)

Feature metadata saved to: /Users/emiliogarcialopez/projectGitHub/telco-churn-project/data/processed/feature_metadata.csv


In [80]:
assert FEATURE_METADATA_PATH.exists()

saved_feature_metadata = pd.read_csv(
    FEATURE_METADATA_PATH
)

assert len(saved_feature_metadata) == X_train_processed_df.shape[1]

assert saved_feature_metadata["feature_name"].tolist() == (
    X_train_processed_df.columns.tolist()
)

print("Feature metadata was saved and validated successfully.")

Feature metadata was saved and validated successfully.


### Feature metadata results

The metadata file was saved successfully and contains the names and positions of all 44 transformed features.

The saved feature order matches the columns in the processed training dataset. This metadata can now be reused during model interpretation, artifact persistence, API development, and future prediction workflows.

## Save fitted preprocessor

The fitted preprocessing object is saved as a reusable model artifact using `joblib`.

The preprocessor contains the transformations learned from `X_train`, including:

- the missing-value imputation strategy;
- the numerical scaling parameters;
- the categorical values learned by the one-hot encoder;
- the final feature transformation structure defined by the `ColumnTransformer`.

Saving this fitted object allows future notebooks and production workflows to apply the same preprocessing rules without fitting them again.

The artifact is stored in the `models` directory as `preprocessor.joblib`.

In [81]:
MODELS_DIR = Path("../models")
PREPROCESSOR_PATH = MODELS_DIR / "preprocessor.joblib"

joblib.dump(preprocessor, PREPROCESSOR_PATH)

print(f"Fitted preprocessor saved to: {PREPROCESSOR_PATH.resolve()}")

Fitted preprocessor saved to: /Users/emiliogarcialopez/projectGitHub/telco-churn-project/models/preprocessor.joblib


### Preprocessor persistence validation

The saved preprocessor is loaded from disk and applied to the original training and test feature sets.

This validation confirms that:

- the artifact can be loaded successfully;
- the loaded preprocessor can transform new data without refitting;
- the transformed training and test outputs preserve the expected dimensions;
- the saved object retains its fitted preprocessing state.

In [82]:
loaded_preprocessor = joblib.load(PREPROCESSOR_PATH)

X_train_check = loaded_preprocessor.transform(X_train)
X_test_check = loaded_preprocessor.transform(X_test)

assert X_train_check.shape == X_train_processed.shape
assert X_test_check.shape == X_test_processed.shape

print("The fitted preprocessor was loaded and validated successfully.")

The fitted preprocessor was loaded and validated successfully.


### Preprocessor persistence results

The fitted preprocessor was saved, loaded, and validated successfully.

The loaded artifact produced transformed training and test datasets with the same dimensions as the original preprocessing results. The fitted-state validation also confirmed that the preprocessor retained the parameters learned from the training data.

This artifact can now be reused during model development. Later, the selected machine learning model and the preprocessor should be combined into a single fitted modeling pipeline for deployment and production predictions.